In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
np.set_printoptions(suppress=True)

df = pd.read_csv("~/Repo/DeepLearning/Class/Churn_Modelling.csv")

I0000 00:00:1792843213.822188 1067921 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792843213.833967 1067921 cuda_fft.cc:470] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
I0000 00:00:1792843213.843301 1067921 cuda_dnn.cc:6964] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
I0000 00:00:1792843213.846536 1067921 cuda_blas.cc:1416] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1792843213.852408 1067921 collection_registry.cc:88] Trying to register 2 metrics with the same name: /coordination_service/agent/enabled. The old value will be erased in order to 

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [3]:
df.head(5)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})
df['GeographyId'] = pd.factorize(df['Geography'])[0]
df.insert(5, 'GeographyId', df.pop('GeographyId'))
df.insert(4, 'CreditScore', df.pop('CreditScore'))
X = df.iloc[:, 4:-1].values
y = df.iloc[:, -1].values
df.tail(5)
# print(X)

,RowNumber,CustomerId,Surname,Geography,CreditScore,GeographyId,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
9995,9996,15606229,Obijiaku,France,771,0,0,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,France,516,0,0,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,France,709,0,1,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,Germany,772,2,0,42,3,75075.31,2,1,0,92888.52,1
9999,10000,15628319,Walker,France,792,0,1,28,4,130142.79,1,1,0,38190.78,0


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

ct = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(), [1])],
    remainder='passthrough'
)
X = ct.fit_transform(X)

In [6]:
print(X[0])

[     1.        0.        0.      619.        1.       42.        2.
      0.        1.        1.        1.   101348.88]


In [7]:
#80% training
#20% testing
from sklearn.model_selection import train_test_split
Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y, test_size=0.2, random_state=1
)
print("Xtrain", Xtrain.shape)
print("ytrain", ytrain.shape)
print("Xtest", Xtest.shape)
print("ytest", ytest.shape)

Xtrain (8000, 12)
ytrain (8000,)
Xtest (2000, 12)
ytest (2000,)


In [8]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
Xtrain = sc.fit_transform(Xtrain)
print(Xtrain)

[[-0.99850112 -0.57273139  1.71490137 ... -1.55337352  0.97725852
   0.42739449]
 [ 1.00150113 -0.57273139 -0.58312392 ... -1.55337352 -1.02327069
  -1.02548708]
 [-0.99850112 -0.57273139  1.71490137 ...  0.64376017  0.97725852
  -0.94479772]
 ...
 [ 1.00150113 -0.57273139 -0.58312392 ...  0.64376017  0.97725852
  -0.14096853]
 [ 1.00150113 -0.57273139 -0.58312392 ...  0.64376017  0.97725852
   0.01781218]
 [-0.99850112 -0.57273139  1.71490137 ...  0.64376017 -1.02327069
  -1.15822478]]


In [9]:
from keras.models import Sequential
from keras.layers import Dense

In [10]:
ann = Sequential()
#units = neuron
ann.add(Dense(units=10, activation='relu'))
ann.add(Dense(units=100, activation='relu'))
ann.add(Dense(units=100, activation='relu'))
ann.add(Dense(units=1, activation='sigmoid'))

In [11]:
ann.compile(optimizer='adam',
            loss='binary_crossentropy',
            metrics=['accuracy'])

I0000 00:00:1792843216.020264 1067921 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9800 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


In [12]:
ann.fit(
    Xtrain,ytrain,batch_size=32,epochs=100)

Epoch 1/100


I0000 00:00:1792843217.127783 1067997 service.cc:153] XLA service 0x7e6bc8034630 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1792843217.127797 1067997 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3060, Compute Capability 8.6 (Driver: 13.2.0; Runtime: 13.4.0; Toolkit: 13.0.0; DNN: 9.27.0)
I0000 00:00:1792843217.145909 1067997 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1792843217.374249 1067997 cuda_dnn.cc:461] Loaded cuDNN version 92700
I0000 00:00:1792843217.378463 1067997 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1749__.9
I0000 00:00:1792843218.934902 1067997 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.


 51/250 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7538 - loss: 0.5986

I0000 00:00:1792843221.029467 1067997 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8031 - loss: 0.4692
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 805us/step - accuracy: 0.8280 - loss: 0.4047
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8430 - loss: 0.3785
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8459 - loss: 0.3677
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8522 - loss: 0.3598
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8541 - loss: 0.3531
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8565 - loss: 0.3486
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 722us/step - accuracy: 0.8600 - loss: 0.3441
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 617us/step - accuracy: 0.8604 - loss: 0.3408
Epoch 10/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 812us/step - accuracy: 0.8621 - loss: 0.3390
Epoch 11/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8620 - loss: 0.3378
Epoch 12/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 

In [13]:
Xtest = sc.transform(Xtest)
print(Xtest)


[[ 1.00150113 -0.57273139 -0.58312392 ...  0.64376017  0.97725852
  -0.05360571]
 [ 1.00150113 -0.57273139 -0.58312392 ...  0.64376017 -1.02327069
  -0.58392685]
 [ 1.00150113 -0.57273139 -0.58312392 ... -1.55337352  0.97725852
  -0.16685331]
 ...
 [-0.99850112  1.74601919 -0.58312392 ... -1.55337352  0.97725852
   1.0669965 ]
 [-0.99850112 -0.57273139  1.71490137 ...  0.64376017  0.97725852
   1.13101314]
 [-0.99850112 -0.57273139  1.71490137 ...  0.64376017  0.97725852
  -0.88790165]]


In [14]:
y_pred = ann.predict(Xtest)
print(y_pred)

63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step
[[0.00084385]
 [0.10051217]
 [0.15600555]
 ...
 [0.0049332 ]
 [0.0003058 ]
 [0.88186795]]


In [15]:
y_pred = (y_pred > 0.5)
print(y_pred)

[[False]
 [False]
 [False]
 ...
 [False]
 [False]
 [ True]]


In [16]:
from sklearn.metrics import accuracy_score
acc = accuracy_score(y_pred, ytest)
print(acc)

0.8405


In [17]:
df.head(1)

,RowNumber,CustomerId,Surname,Geography,CreditScore,GeographyId,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,France,619,0,1,42,2,0.0,1,1,1,101348.88,1


In [24]:
df[df['Geography'] == 'France']['GeographyId'].unique()

array([0])

In [25]:
new_customer = [[600, 0, 0, 40, 3, 60000, 2, 1, 1, 50000]]

new_customer = ct.transform(new_customer)
new_customer = sc.transform(new_customer)

In [26]:
prediction_prob = ann.predict(new_customer)

print("Probability of exiting:", prediction_prob[0][0])
print("Will the customer exit?", (prediction_prob > 0.5)[0][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Probability of exiting: 0.005564059
Will the customer exit? False
